In [1]:
import os
import numpy as np
import pandas as pd
import dearpygui.dearpygui as dpg
import nd2
import cv2
from skimage import exposure, measure
from scipy.ndimage import center_of_mass
from scipy.stats import skew
from scipy.signal import find_peaks
from skimage.measure import label, regionprops
from cellpose import models, denoise

In [2]:
data = pd.read_csv('one stack_raw.csv')

def convert_list_strings_to_lists(df, columns=None):
    df_copy = df.copy()
    cols_to_check = columns or df_copy.columns

    for col in cols_to_check:
        if df_copy[col].dtype == object:
            def try_convert(val):
                if isinstance(val, str):
                    val = val.strip()
                    if val.startswith("[") and val.endswith("]"):
                        try:
                            return np.fromstring(val.strip("[]"), sep=" ").tolist()
                        except Exception:
                            return val
                    elif "," in val:
                        try:
                            return [float(x) for x in val.split(",")]
                        except Exception:
                            return val
                return val
            df_copy[col] = df_copy[col].apply(try_convert)

    return df_copy


data = convert_list_strings_to_lists(data)
data

,mask_id,Slice_Seperation,X_vals,file_name,DJID,Sex,Eye,Time_Min,eGFP_Value,eGFP_Raw_Intensity,in_rip,Y_vals_DAPI,Y_vals_eGFP,Y_vals_WGA,Y_vals_GLUT1,original_mask_id
0,0,0.15,"[0.0, 0.15, 0.3, 0.44999999999999996, 0.6, 0.7...",2007R_GLUT1_647_WGA_594_0002.nd2,2007,M,R,5,False,3.063058,False,"[12.70017847, 12.66745985, 12.70999405, 12.855...","[1.82004759, 1.85663296, 1.88548483, 1.9039262...","[5.62581797, 5.49702558, 5.88072576, 6.4283164...","[0.57049375, 0.57198096, 0.6154075, 0.70523498...",0
1,1,0.15,"[0.0, 0.15, 0.3, 0.44999999999999996, 0.6, 0.7...",2007R_GLUT1_647_WGA_594_0002.nd2,2007,M,R,5,False,3.536082,False,"[13.10790378, 13.20389462, 13.38258877, 13.540...","[1.82542955, 1.84444444, 1.86620848, 1.8474226...","[5.26071019, 5.19747995, 5.49415808, 5.9585337...","[0.52119129, 0.51592211, 0.54776632, 0.6568155...",1
2,2,0.15,"[0.0, 0.15, 0.3, 0.44999999999999996, 0.6, 0.7...",2007R_GLUT1_647_WGA_594_0002.nd2,2007,M,R,5,False,12.084552,False,"[13.24528051, 13.36134007, 13.48869981, 13.736...","[2.02087211, 2.04187716, 2.08867323, 2.0937250...","[5.04573252, 5.00505185, 5.3126828, 5.75565009...","[0.66192502, 0.65128955, 0.73384738, 0.8546929...",2
3,3,0.15,"[0.0, 0.15, 0.3, 0.44999999999999996, 0.6, 0.7...",2007R_GLUT1_647_WGA_594_0002.nd2,2007,M,R,5,False,3.425488,False,"[12.78108941, 12.91264132, 12.82733813, 13.180...","[1.96608428, 2.04110997, 2.04727646, 1.9578622...","[5.12230216, 5.0668037, 5.2672148, 5.7954779, ...","[0.49537513, 0.52312436, 0.55498458, 0.6649537...",4
4,4,0.15,"[0.0, 0.15, 0.3, 0.44999999999999996, 0.6, 0.7...",2007R_GLUT1_647_WGA_594_0002.nd2,2007,M,R,5,False,3.427007,False,"[13.11678832, 13.26277372, 13.19708029, 13.540...","[1.86131387, 1.88321168, 1.95985401, 1.9525547...","[5.60583942, 5.36131387, 5.60948905, 6.0474452...","[0.62773723, 0.51459854, 0.54014599, 0.7080292...",7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,119,0.15,"[0.0, 0.15, 0.3, 0.44999999999999996, 0.6, 0.7...",2007R_GLUT1_647_WGA_594_0002.nd2,2007,M,R,5,True,84.190963,False,"[12.97288802, 13.23497053, 13.47308448, 13.594...","[2.12717747, 2.1962017, 2.21139489, 2.24426981...","[4.99423707, 5.40497708, 5.71486575, 6.1250818...","[0.48827767, 0.57066143, 0.63392272, 0.7275704...",124
120,120,0.15,"[0.0, 0.15, 0.3, 0.44999999999999996, 0.6, 0.7...",2007R_GLUT1_647_WGA_594_0002.nd2,2007,M,R,5,False,4.997976,False,"[13.51272412, 13.88548294, 14.16252169, 14.574...","[1.97252747, 1.9832273, 2.0242915, 2.0373048, ...","[4.77096588, 5.09774436, 5.40746096, 5.7411798...","[0.46790052, 0.56072874, 0.61336032, 0.7038750...",125
121,121,0.15,"[0.0, 0.15, 0.3, 0.44999999999999996, 0.6, 0.7...",2007R_GLUT1_647_WGA_594_0002.nd2,2007,M,R,5,True,55.502388,False,"[12.59948269, 12.90370076, 12.88460008, 13.072...","[2.07282133, 2.08714684, 2.15419817, 2.1541981...","[5.05212893, 5.49641862, 5.85614803, 6.2379625...","[0.5051731, 0.59112614, 0.67449264, 0.78830084...",126
122,122,0.15,"[0.0, 0.15, 0.3, 0.44999999999999996, 0.6, 0.7...",2007R_GLUT1_647_WGA_594_0002.nd2,2007,M,R,5,False,3.017331,False,"[14.35112652, 14.95771231, 15.91681109, 16.420...","[1.92443674, 1.93414211, 1.96533795, 1.9493934...","[4.6135182, 4.96291161, 5.19237435, 5.52166378...","[0.44367418, 0.54280763, 0.64714038, 0.705026,...",127


In [22]:
def run_integral_analysis(trace_data_df):
    df = trace_data_df.copy()

    print(f"[DEBUG] Starting analysis with {len(df)} rows")

    # Add separation and cell identity
    df["Cell"] = df["file_name"].astype(str) + "_mask" + df["mask_id"].astype(str)

    # Peak detection
    df = WGA_Peaks_Finder_V2(df)
    print(f"[DEBUG] After WGA_Peaks_Finder_V2: {len(df)} rows")
    print("[DEBUG] Sample DAPI_peak_index values:")
    print(df["DAPI_peak_index"].head(10))
    print(df["DAPI_peak_index"].apply(type).value_counts())


    df = filter_out_unclear_DAPI(df)
    print(f"[DEBUG] After filter_out_unclear_DAPI: {len(df)} rows")

    if len(df) == 0:
        print("[ERROR] No valid cells remaining after DAPI filtering.")
        return df

    # Integrals
    df = Top_Bottom_Indices_V2(df)
    df = TopMidBot_Integrals_V2(df)
    df = Surface_Integrals_V2(df)

    df = Replace_NaNs_With_None(df)

    print(f"[DEBUG] Final dataframe shape: {df.shape}")
    return df

def WGA_Peaks_Finder_V2(dataframe, prom_val: float = 1.0):
    """
    Identifies WGA peaks before and after a single DAPI peak for each row.
    Adds:
    - WGA_Middle_Indices: [peak_before_dapi, peak_after_dapi]
    - DAPI_peak_index: index of peak in DAPI channel
    - Length: distance between WGA peaks in microns
    - Cell: integer ID
    """
    wga_middle = []
    dapi_peaks = []
    lengths = []
    cell_ids = []

    for idx, row in dataframe.iterrows():
        y_wga = row.get("Y_vals_WGA", [])
        y_dapi = row.get("Y_vals_DAPI", [])
        sep = row.get("Slice_Seperation", np.nan)

        print(f"[DEBUG] Cell index: {idx}")
        print(f"[DEBUG] y_wga type: {type(y_wga)}, len: {len(y_wga) if hasattr(y_wga, '__len__') else 'N/A'}")
        print(f"[DEBUG] y_dapi type: {type(y_dapi)}, len: {len(y_dapi) if hasattr(y_dapi, '__len__') else 'N/A'}")
        print(f"[DEBUG] sep: {sep}")

        dapi_dist = int(12 / sep)
        wga_dist = int(1.05 / sep)

        dapi_indices, _ = find_peaks(y_dapi, prominence=prom_val, distance=dapi_dist)
        print('DEBUG', dapi_indices)
        wga_indices, _ = find_peaks(y_wga, prominence=prom_val, distance=wga_dist)

        peak_before = np.nan
        peak_after = np.nan

        if len(dapi_indices) == 1:
            dapi_idx = dapi_indices[0]
            for peak in wga_indices:
                if peak < dapi_idx:
                    peak_before = peak
                elif peak > dapi_idx and np.isnan(peak_after):
                    peak_after = peak
                    break
        else:
            dapi_idx = np.nan

        dist = (peak_after - peak_before) * sep if not np.isnan(peak_before) and not np.isnan(peak_after) else np.nan

        wga_middle.append([peak_before, peak_after])
        dapi_peaks.append(dapi_idx)
        lengths.append(dist)
        cell_ids.append(idx)

    dataframe["WGA_Middle_Indices"] = wga_middle
    dataframe["DAPI_peak_index"] = dapi_peaks
    dataframe["Length"] = lengths
    dataframe["Cell"] = cell_ids

    return dataframe

def filter_out_unclear_DAPI(dataframe):
    """
    Keeps rows where 'DAPI_peak_index' is a valid number (not NaN or None).
    Prints out the number and identities of filtered-out cells for debugging.
    """

    valid_rows = dataframe[dataframe["DAPI_peak_index"].apply(lambda x: pd.notna(x) and isinstance(x, (int, float)))].copy()
    filtered_out = dataframe[~dataframe.index.isin(valid_rows.index)]

    if not filtered_out.empty:
        print("Filtered out cells (no valid DAPI peak):", filtered_out["Cell"].unique().tolist())
    else:
        print("No cells were filtered out.")

    return valid_rows.reset_index(drop=True)

def Top_Bottom_Indices_V2(dataframe, microns_extension: float = 1.5):
    '''
    Calculates WGA_Top_Indices and WGA_Bottom_Indices based on Slice_Seperation and WGA_Middle_Indices.
    '''
    grouped = dataframe.groupby('Cell')
    slice_separation = grouped['Slice_Seperation'].first()
    first_peaks = grouped['WGA_Middle_Indices'].apply(lambda x: x.iloc[0] if len(x) > 0 else [np.nan, np.nan])

    index_offset = (microns_extension / slice_separation).fillna(0).astype(int)

    l_middle = first_peaks.apply(lambda x: x[0] if len(x) > 0 else np.nan)
    r_middle = first_peaks.apply(lambda x: x[1] if len(x) > 1 else np.nan)

    l_top = np.maximum(l_middle - index_offset, 0)
    r_bot = r_middle + index_offset

    r_middle = r_middle.apply(lambda x: None if pd.isna(x) else x)
    r_bot = r_bot.apply(lambda x: None if pd.isna(x) else x)

    idx_df = pd.DataFrame({
        'Cell': grouped.size().index,
        'WGA_Top_Indices': list(zip(l_top, l_middle)),
        'WGA_Bottom_Indices': list(zip(r_middle, r_bot))
    })

    dataframe["WGA_Top_Indices"] = list(zip(l_top, l_middle))
    dataframe["WGA_Bottom_Indices"] = list(zip(r_middle, r_bot))
    return dataframe

def TopMidBot_Integrals_V2(dataframe):
    """
    Calculates WGA Top, Middle, Bottom integrals using defined index pairs.
    Adds columns: WGA_Top_Integral, WGA_Middle_Integral, WGA_Bottom_Integral
    """
    def integral_calculator(y_vals, indices):
        if not isinstance(indices, (list, tuple)) or pd.isna(indices[0]) or pd.isna(indices[1]):
            return None
        try:
            start_idx, end_idx = int(indices[0]), int(indices[1])
            start_idx = max(start_idx, 0)
            end_idx = min(end_idx, len(y_vals))
            if start_idx >= end_idx:
                return None
            return float(np.sum(np.array(y_vals)[start_idx:end_idx]))
        except:
            return None

    for section in ['Middle', 'Top', 'Bottom']:
        col_name = f"WGA_{section}_Integral"
        index_col = f"WGA_{section}_Indices"
        dataframe[col_name] = dataframe.apply(
            lambda row: integral_calculator(row.get('Y_vals_WGA', []), row.get(index_col)), axis=1
        )

    return dataframe

def Surface_Integrals_V2(dataframe):
    def compute_surface(row):
        peak_indices = row.get("WGA_Middle_Indices", [np.nan, np.nan])
        x_vals = row.get("X_vals", [])
        y_G = row.get("Y_vals_GLUT1", [])
        y_W = row.get("Y_vals_WGA", [])
        sep = row.get("Slice_Seperation", None)
        idx_offset = int(1.5 / sep) if sep else 3

        def get_integral(idx, y_vals):
            if pd.isna(idx):
                return np.nan
            idx = int(idx)
            left = max(idx - idx_offset, 0)
            right = min(idx + idx_offset, len(x_vals))
            return np.sum(y_vals[left:right])

        top_G = get_integral(peak_indices[0], y_G)
        bot_G = get_integral(peak_indices[1], y_G)
        top_W = get_integral(peak_indices[0], y_W)
        bot_W = get_integral(peak_indices[1], y_W)

        return pd.Series({
            "GluT1_Top_Surface_Integral": top_G,
            "GluT1_Bot_Surface_Integral": bot_G,
            "WGA_Top_Surface_Integral": top_W,
            "WGA_Bot_Surface_Integral": bot_W,
            "Top_Surface_Ratio": top_G / top_W if not pd.isna(top_G) and not pd.isna(top_W) and top_W != 0 else np.nan,
            "Bot_Surface_Ratio": bot_G / bot_W if not pd.isna(bot_G) and not pd.isna(bot_W) and bot_W != 0 else np.nan,
        })
    
    surface_df = dataframe.apply(compute_surface, axis=1)
    for col in surface_df.columns:
        dataframe[col] = surface_df[col]
    return dataframe

def Replace_NaNs_With_None(dataframe):
    """
    Replaces all `NaN` values in a DataFrame with `None`, including those inside lists and tuples.
    """
    def replace_in_iterable(iterable):
        return type(iterable)(None if pd.isna(item) else item for item in iterable)

    def replace_nans(item):
        if isinstance(item, (list, tuple)):
            return replace_in_iterable(item)
        elif pd.isna(item):
            return None
        else:
            return item

    return dataframe.applymap(replace_nans)

final = run_integral_analysis(data)

[DEBUG] Starting analysis with 124 rows
[DEBUG] Cell index: 0
[DEBUG] y_wga type: <class 'list'>, len: 102
[DEBUG] y_dapi type: <class 'list'>, len: 102
[DEBUG] sep: 0.15
DEBUG [46]
[DEBUG] Cell index: 1
[DEBUG] y_wga type: <class 'list'>, len: 102
[DEBUG] y_dapi type: <class 'list'>, len: 102
[DEBUG] sep: 0.15
DEBUG [38]
[DEBUG] Cell index: 2
[DEBUG] y_wga type: <class 'list'>, len: 102
[DEBUG] y_dapi type: <class 'list'>, len: 102
[DEBUG] sep: 0.15
DEBUG [29]
[DEBUG] Cell index: 3
[DEBUG] y_wga type: <class 'list'>, len: 102
[DEBUG] y_dapi type: <class 'list'>, len: 102
[DEBUG] sep: 0.15
DEBUG [42]
[DEBUG] Cell index: 4
[DEBUG] y_wga type: <class 'list'>, len: 102
[DEBUG] y_dapi type: <class 'list'>, len: 102
[DEBUG] sep: 0.15
DEBUG [37]
[DEBUG] Cell index: 5
[DEBUG] y_wga type: <class 'list'>, len: 102
[DEBUG] y_dapi type: <class 'list'>, len: 102
[DEBUG] sep: 0.15
DEBUG [37]
[DEBUG] Cell index: 6
[DEBUG] y_wga type: <class 'list'>, len: 102
[DEBUG] y_dapi type: <class 'list'>, len: